# 29 — Evaluation RAG

> **Tier 7 | Production**

## What is Evaluation RAG?

Building a RAG pipeline is only half the job. **Evaluation RAG** measures whether the
system actually works — both the retrieval stage and the generation stage — using
quantitative metrics.

### Metrics covered

| Stage | Metric | Definition |
|-------|--------|------------|
| Retrieval | **Hit\@K** | Was the source chunk in top-K results? |
| Retrieval | **MRR** | Mean Reciprocal Rank — how high did the source chunk rank? |
| Retrieval | **NDCG\@K** | Normalized Discounted Cumulative Gain |
| Generation | **Faithfulness** | Are answer claims supported by context? (LLM judge, 1–5) |
| Generation | **Answer Relevance** | Does the answer address the question? (LLM judge, 1–5) |
| System | **Latency** | Embed / retrieve / generate breakdown |

### Evaluation dataset
Ground-truth Q&A pairs are **generated synthetically**: we sample chunks from the index,
ask an LLM to write a question answerable *only* from that chunk, and record the source
chunk as the relevant document. This gives us a labelled eval set without human annotation.

## PDF Framework: pdfplumber `extract_text()`

NB 24 used pdfplumber's `extract_words()` with manual line grouping. This notebook uses
pdfplumber's built-in `extract_text()` string method — simpler, single call per page.

| pdfplumber method | Output | Used in |
|------------------|--------|--------|
| `extract_words()` | List of word dicts with coordinates | NB 24 |
| **`extract_text()`** | **Plain string** | **NB 29** |


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pdfplumber extract_text"]
        PDF(["📄 climate.pdf"])
        PDF --> PL["pdfplumber\nextract_text()"]
        PL --> CH["Text chunks"]
        CH --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\neval_rag_29")]
    end
    subgraph EVALSET ["🟡  EVAL DATASET GENERATION"]
        SC["Sample N chunks"]
        SC --> QGEN["LLM: generate question\nfrom each chunk"]
        QGEN --> ES[("Eval set\n(question, source_chunk)")]
    end
    subgraph EVAL ["🟢  EVALUATION"]
        ES --> RET["Retrieve top-K"]
        RET --> RMET["Hit@K · MRR · NDCG"]
        ES --> GEN["Full RAG answer"]
        GEN --> GMET["Faithfulness · Relevance\n(LLM judge)"]
        RMET --> RPT(["📊 Report"])
        GMET --> RPT
    end
    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style EVALSET fill:#fef9c3,stroke:#ca8a04,color:#713f12
    style EVAL fill:#dcfce7,stroke:#16a34a,color:#14532d
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "pdfplumber", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, math
from typing import List, Dict, Optional
from dotenv import load_dotenv

import boto3
import pdfplumber
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")
print(f"pdfplumber version: {pdfplumber.__version__}")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

COLLECTION_NAME = "eval_rag_29"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
TOP_K           = 5
N_EVAL          = 6    # number of eval questions to generate

PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection : {COLLECTION_NAME}")
print(f"PDF        : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"N_EVAL     : {N_EVAL} questions")


## Step 3 — Qdrant Setup

In [ ]:
def make_qdrant(url='', api_key=''):
    if url:
        try:
            kw = {'url': url}
            if api_key: kw['api_key'] = api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {url}')
            return client
        except Exception as e:
            print(f'Qdrant Cloud unavailable ({e}), falling back...')
    print('Using Qdrant in-memory')
    return QdrantClient(':memory:')


qdrant = make_qdrant(QDRANT_URL, QDRANT_API_KEY)

if COLLECTION_NAME in [c.name for c in qdrant.get_collections().collections]:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
print(f'Created "{COLLECTION_NAME}" (dim={EMBEDDING_DIM})')


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)


def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]


def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out


def call_llm(system: str, user_content: str, max_tokens: int = 512) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user_content}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]


test_emb = embed_text("evaluation rag pdfplumber test")
print(f"Embedding OK — dim={len(test_emb)}")


## Step 5 — PDF Loading with pdfplumber `extract_text()`

pdfplumber's `extract_text()` returns a plain string per page by analysing the PDF's
character positions and inserting spaces/newlines where gaps appear. It is simpler than
the `extract_words()` approach (NB 24) — one call per page, no manual grouping needed.


In [ ]:
def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def load_pdf_pdfplumber_text(path: str):
    chunks = []
    with pdfplumber.open(path) as pdf:
        n_pages = len(pdf.pages)
        for page_num, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            text = ' '.join(text.split()).strip()
            if not text:
                continue
            for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
                chunks.append({
                    'text'      : sub,
                    'page'      : page_num + 1,
                    'char_count': len(sub),
                })
    stats = {
        'n_pages' : n_pages,
        'n_chunks': len(chunks),
        'avg_chars': sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
    }
    return chunks, stats


t0 = time.time()
chunks, stats = load_pdf_pdfplumber_text(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pdfplumber extract_text : {elapsed:.0f}ms")
print(f"Pages                   : {stats['n_pages']}")
print(f"Chunks                  : {stats['n_chunks']}")
print(f"Avg chunk length        : {stats['avg_chars']} chars")


## Step 6 — Index

In [ ]:
print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

pts = [
    PointStruct(
        id=str(uuid.uuid4()), vector=embs[i],
        payload={
            'text'    : chunks[i]['text'],
            'metadata': {
                'page'      : chunks[i]['page'],
                'char_count': chunks[i]['char_count'],
                'source'    : 'climate.pdf',
                'pdf_lib'   : 'pdfplumber-text',
            }
        }
    )
    for i in range(len(chunks))
]
qdrant.upsert(collection_name=COLLECTION_NAME, points=pts)
print(f"Indexed {qdrant.get_collection(COLLECTION_NAME).points_count} vectors in {time.time()-t0:.1f}s")


## Step 7 — Build Evaluation Dataset

We sample `N_EVAL` chunks spread evenly across the document, then ask the LLM to write
one specific factual question per chunk that can *only* be answered from that chunk.
The (question, source_chunk) pair is our ground-truth eval entry.


In [ ]:
QGEN_SYSTEM = "You create evaluation questions for RAG systems. Be precise and concise."

QGEN_PROMPT = """Given this passage from a climate document, write ONE specific factual question
that can be answered ONLY from this passage. The question must target a concrete fact,
number, name, or definition present in the text.

Passage:
{chunk}

Return ONLY the question — no explanation, no prefix."""


def generate_eval_question(chunk_text: str) -> str:
    return call_llm(
        QGEN_SYSTEM,
        QGEN_PROMPT.format(chunk=chunk_text),
        max_tokens=80,
    ).strip()


# Sample N_EVAL chunks evenly across the document
step = max(1, len(chunks) // N_EVAL)
sampled = [chunks[i * step] for i in range(N_EVAL)]

print(f"Generating {N_EVAL} eval questions from sampled chunks...")
eval_set = []
for idx, c in enumerate(sampled):
    q = generate_eval_question(c['text'])
    eval_set.append({
        'question'    : q,
        'source_chunk': c['text'],
        'page'        : c['page'],
        'chunk_id'    : c['text'][:80].strip(),   # identity key
    })
    print(f"  [{idx+1}/{N_EVAL}] p{c['page']:>2}: {q}")

print(f"\nEval set built: {len(eval_set)} questions")


## Step 8 — Retrieval Evaluation

For each eval question we retrieve top-K chunks and check whether the source chunk
appears in the results. Three metrics:

| Metric | Formula |
|--------|---------|
| **Hit\@K** | 1 if source chunk in top-K, else 0 |
| **MRR** | 1/rank if found, else 0 |
| **NDCG\@K** | DCG / IDCG where relevance is binary (1=source chunk) |


In [ ]:
def compute_ndcg(relevance: List[int]) -> float:
    dcg  = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))
    idcg = 1.0   # ideal: source chunk at rank 1
    return dcg / idcg


def evaluate_retrieval(eval_set: List[Dict], top_k: int = TOP_K) -> List[Dict]:
    results = []
    for item in eval_set:
        qvec = embed_text(item['question'])
        resp = qdrant.query_points(
            collection_name=COLLECTION_NAME, query=qvec,
            limit=top_k, with_payload=True)
        retrieved_texts = [p.payload['text'] for p in resp.points]
        retrieved_ids   = [t[:80].strip() for t in retrieved_texts]

        source_id = item['chunk_id']
        relevance = [1 if rid == source_id else 0 for rid in retrieved_ids]

        hit  = int(source_id in retrieved_ids)
        rank = next((i + 1 for i, r in enumerate(relevance) if r), None)
        mrr  = 1.0 / rank if rank else 0.0
        ndcg = compute_ndcg(relevance)

        results.append({
            'question': item['question'],
            'page'    : item['page'],
            'hit'     : hit,
            'rank'    : rank,
            'mrr'     : mrr,
            'ndcg'    : ndcg,
        })
        time.sleep(0.05)
    return results


print(f"Evaluating retrieval over {len(eval_set)} questions (top_k={TOP_K})...")
ret_results = evaluate_retrieval(eval_set)

print(f"\n{'Question':<48} {'Hit':>4} {'Rank':>5} {'MRR':>6} {'NDCG':>6}")
print("-" * 72)
for r in ret_results:
    rank_str = str(r['rank']) if r['rank'] else '-'
    print(f"{r['question'][:47]:<48} {r['hit']:>4} {rank_str:>5} {r['mrr']:>6.3f} {r['ndcg']:>6.3f}")

avg_hit  = sum(r['hit']  for r in ret_results) / len(ret_results)
avg_mrr  = sum(r['mrr']  for r in ret_results) / len(ret_results)
avg_ndcg = sum(r['ndcg'] for r in ret_results) / len(ret_results)

print("-" * 72)
print(f"{'Average':<48} {avg_hit:>4.2f} {'':>5} {avg_mrr:>6.3f} {avg_ndcg:>6.3f}")
print(f"\nHit@{TOP_K}={avg_hit:.2f}  MRR={avg_mrr:.3f}  NDCG@{TOP_K}={avg_ndcg:.3f}")


## Step 9 — Generation Evaluation (LLM Judge)

For each eval question we run the full RAG pipeline and evaluate the answer on two
dimensions using the LLM as a judge:

- **Faithfulness**: are all claims in the answer grounded in the retrieved context?
- **Answer Relevance**: does the answer actually address the question?

Both scored 1–5. The judge is prompted to return structured JSON.


In [ ]:
SYSTEM_RAG = """You are a knowledgeable assistant answering questions about a climate document.
Answer ONLY from the provided context. Be concise and precise.
If the answer is not in the context, say so explicitly.
"""

JUDGE_SYSTEM = "You are a strict evaluator of RAG system quality. Return valid JSON only."

FAITHFULNESS_TMPL = """Rate the faithfulness of this answer on a 1-5 scale.
5=all claims directly supported by context, 1=answer contradicts or ignores context.

Context:
{context}

Answer:
{answer}

Return ONLY valid JSON: {{"score": <1-5>, "reason": "<one sentence>"}}"""

RELEVANCE_TMPL = """Rate how well this answer addresses the question on a 1-5 scale.
5=completely and directly answers the question, 1=does not address the question.

Question: {question}

Answer: {answer}

Return ONLY valid JSON: {{"score": <1-5>, "reason": "<one sentence>"}}"""


def rag_answer(question: str):
    qvec = embed_text(question)
    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME, query=qvec,
        limit=TOP_K, with_payload=True)
    hits    = [p.payload['text'] for p in resp.points]
    context = '\n\n'.join(f"[{i+1}]\n{h}" for i, h in enumerate(hits))
    answer  = call_llm(SYSTEM_RAG, f"Context:\n{context}\n\nQuestion: {question}",
                       max_tokens=512)
    return answer, context


def judge_score(raw: str) -> Dict:
    try:
        start = raw.find('{')
        end   = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except Exception:
        return {'score': 0, 'reason': 'parse error'}


print(f"Running generation evaluation on {len(eval_set)} questions...")
gen_results = []

for item in eval_set:
    answer, context = rag_answer(item['question'])

    faith_raw = call_llm(JUDGE_SYSTEM,
                         FAITHFULNESS_TMPL.format(context=context[:2000], answer=answer),
                         max_tokens=120)
    relev_raw = call_llm(JUDGE_SYSTEM,
                         RELEVANCE_TMPL.format(question=item['question'], answer=answer),
                         max_tokens=120)

    faith = judge_score(faith_raw)
    relev = judge_score(relev_raw)

    gen_results.append({
        'question'   : item['question'],
        'answer'     : answer,
        'faithfulness': faith['score'],
        'faith_reason': faith['reason'],
        'relevance'  : relev['score'],
        'relev_reason': relev['reason'],
    })
    print(f"  Q: {item['question'][:60]}")
    print(f"     faithfulness={faith['score']}/5  relevance={relev['score']}/5")

avg_faith = sum(r['faithfulness'] for r in gen_results) / len(gen_results)
avg_relev = sum(r['relevance']    for r in gen_results) / len(gen_results)
print(f"\nAvg Faithfulness : {avg_faith:.2f}/5")
print(f"Avg Relevance    : {avg_relev:.2f}/5")


## Step 10 — Latency Benchmark

Break total RAG latency into three stages: embed, retrieve, generate.


In [ ]:
print(f"Latency benchmark over {len(eval_set)} questions...")
lat_results = []

for item in eval_set:
    t0 = time.time()

    qvec     = embed_text(item['question'])
    t_embed  = (time.time() - t0) * 1000

    resp     = qdrant.query_points(
        collection_name=COLLECTION_NAME, query=qvec,
        limit=TOP_K, with_payload=True)
    t_ret    = (time.time() - t0) * 1000 - t_embed

    hits     = [p.payload['text'] for p in resp.points]
    context  = '\n\n'.join(f"[{i+1}]\n{h}" for i, h in enumerate(hits))
    _        = call_llm(SYSTEM_RAG, f"Context:\n{context}\n\nQuestion: {item['question']}",
                        max_tokens=512)
    t_llm    = (time.time() - t0) * 1000 - t_embed - t_ret
    t_total  = (time.time() - t0) * 1000

    lat_results.append({
        'embed_ms'   : t_embed,
        'retrieve_ms': t_ret,
        'llm_ms'     : t_llm,
        'total_ms'   : t_total,
    })

def avg(lst): return sum(lst) / len(lst) if lst else 0

print(f"\n{'Stage':<12} {'Avg (ms)':>10} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 45)
for stage, key in [("Embed", 'embed_ms'), ("Retrieve", 'retrieve_ms'),
                   ("LLM", 'llm_ms'), ("Total", 'total_ms')]:
    vals = [r[key] for r in lat_results]
    print(f"{stage:<12} {avg(vals):>10.0f} {min(vals):>10.0f} {max(vals):>10.0f}")


## Step 11 — Summary Report

In [ ]:
print("=" * 60)
print("EVALUATION REPORT — Notebook 29")
print("=" * 60)
print(f"Collection  : {COLLECTION_NAME}")
print(f"Vectors     : {qdrant.get_collection(COLLECTION_NAME).points_count}")
print(f"Eval set    : {len(eval_set)} questions (LLM-generated, synthetic ground truth)")
print(f"PDF library : pdfplumber extract_text() — plain string per page")
print()
print("Retrieval metrics")
print("-" * 30)
print(f"  Hit@{TOP_K}  : {avg_hit:.2f}  ({sum(r['hit'] for r in ret_results)}/{len(ret_results)} questions)")
print(f"  MRR     : {avg_mrr:.3f}")
print(f"  NDCG@{TOP_K} : {avg_ndcg:.3f}")
print()
print("Generation metrics (LLM judge, 1–5)")
print("-" * 30)
print(f"  Faithfulness : {avg_faith:.2f}/5")
print(f"  Relevance    : {avg_relev:.2f}/5")
print()
print("Latency (avg)")
print("-" * 30)
print(f"  Embed    : {avg([r['embed_ms']    for r in lat_results]):.0f}ms")
print(f"  Retrieve : {avg([r['retrieve_ms'] for r in lat_results]):.0f}ms")
print(f"  LLM      : {avg([r['llm_ms']      for r in lat_results]):.0f}ms")
print(f"  Total    : {avg([r['total_ms']     for r in lat_results]):.0f}ms")
print()
print("Notebook 29 — Evaluation RAG complete.")


## Summary

### Evaluation framework

| Component | Approach |
|-----------|----------|
| Ground truth | Synthetic — LLM generates Q from chunk, chunk is the relevant doc |
| Retrieval eval | Hit@K, MRR, NDCG@K — chunk identity by text prefix |
| Generation eval | LLM-as-judge: faithfulness and answer relevance, 1–5 scale |
| Latency | Embed / retrieve / generate stage breakdown |

### LLM-as-judge prompting pattern

```python
# Faithfulness: does the answer stay within the context?
judge_prompt = "Rate faithfulness 1-5 ... Return ONLY JSON: {score, reason}"
raw = call_llm(JUDGE_SYSTEM, judge_prompt)
score = json.loads(raw[raw.find('{'):raw.rfind('}')+1])['score']
```

### pdfplumber modes

| Method | Output type | Grouping |
|--------|------------|----------|
| `extract_text()` | Plain string | Automatic |
| `extract_words()` | List of word dicts | Manual (NB 24) |

> **Next: [30 — Complete Pipeline RAG](30_Complete_Pipeline_RAG.ipynb)** —
> end-to-end production pipeline combining indexing, retrieval, caching, streaming, and evaluation.
